# Thu thập dữ liệu Art Institute of Chicago cho CDNMF

Notebook gồm hai bước: crawl tác phẩm từ API Art Institute of Chicago, sau đó xuất dữ liệu CDNMF. Mỗi node là một tác phẩm (ID `id`); các tác phẩm cùng `artist_id` được nối với tối đa 20 láng giềng. Nhãn là `artwork_type_title`; `title` và `artist_title` chỉ lưu làm thông tin, không dùng làm feature.

Chạy cell crawl trước. Nếu quá trình bị ngắt, chạy lại cell đó để tiếp tục từ checkpoint.

In [ ]:
from pathlib import Path
from crawl_artic_cdnmf import crawl_artworks

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "Database" / "artic_25000"
RAW_CSV = OUTPUT_DIR / "artworks_raw.csv"
CRAWL_LIMIT = 25_000

if not (ROOT / "crawl_artic_cdnmf.py").is_file():
    raise FileNotFoundError("Hãy mở notebook từ thư mục gốc repository CDNMF.")

collected_count = crawl_artworks(CRAWL_LIMIT, RAW_CSV)
print(f"Đã lưu {collected_count:,} artwork đầy đủ trường vào {RAW_CSV}")
print(f"Checkpoint: {RAW_CSV.with_suffix('.checkpoint.json')}")

## 2. Tiền xử lý CSV

Dữ liệu sau xử lý giữ tối đa 25.000 artwork hợp lệ. Không giới hạn số department, không lọc nhãn ít mẫu và không cân bằng số lượng theo nhãn; `group.txt` chứa nhãn cho mọi department có trong dữ liệu. Các tác phẩm của mọi tác giả đều được giữ lại.

Feature mặc định gồm `material_titles`, `technique_titles`, `place_of_origin`, `medium_display`; thêm trường có độ phủ cao hơn giữa `subject_titles`/`style_titles` nếu đạt ít nhất 50%. `title`, `artist_title` và `artwork_type_title` không nằm trong feature. Graph nối các tác phẩm cùng tác giả, tối đa 20 láng giềng mỗi node; số cộng đồng k bằng số lớp còn lại.

In [ ]:
from crawl_artic_cdnmf import prepare_dataset, read_artwork_csv

PROCESSED_LIMIT = 25_000
FEATURE_LIMIT = 128
NEIGHBORS = 20
OPTIONAL_FEATURE_COVERAGE = 0.5

if not RAW_CSV.is_file():
    raise FileNotFoundError("Chạy cell thu thập dữ liệu trước để tạo artworks_raw.csv.")

result = prepare_dataset(
    records=read_artwork_csv(RAW_CSV, CRAWL_LIMIT),
    output_dir=OUTPUT_DIR,
    processed_limit=PROCESSED_LIMIT,
    label_field="artwork_type_title",
    feature_limit=FEATURE_LIMIT,
    neighbors=NEIGHBORS,
    optional_feature_coverage=OPTIONAL_FEATURE_COVERAGE,
)
print(result)
print(f"CSV gốc: {RAW_CSV}")
print(f"CSV sau tiền xử lý: {OUTPUT_DIR / 'artworks.csv'}")